# 2. Trajectories

`crd_convert` is the workhorse for reading trajectories. It reads DCD/PDB/AMBER/
GROMACS files, optionally applies an atom **selection**, structural **fitting**,
and **centering**, and returns coordinates as a NumPy array ready for analysis.

```{admonition} Tuple return value
:class: important
`crd_convert` returns `(list_of_trajectories, subset_molecule)`. The subset
molecule contains only the atoms picked by `selection`, and its atom ordering
matches the coordinates in the returned trajectories.
```


In [ ]:
import numpy as np
from genepie import genesis_exe, SMolecule
from genepie.tests.conftest import BPTI_PDB, BPTI_PSF, BPTI_DCD

mol = SMolecule.from_file(pdb=BPTI_PDB, psf=BPTI_PSF)

## Basic load

With `selection="all"`, every atom is kept.

In [ ]:
trajs, subset = genesis_exe.crd_convert(
    mol,
    trj_files=[str(BPTI_DCD)],
    trj_format="DCD",
    trj_type="COOR+BOX",
    selection="all",
)
traj = trajs[0]
print("coords shape (nframe, natom, 3):", traj.coords.shape)
print("PBC boxes (nframe, 3, 3)      :", traj.pbc_boxes.shape)
print("subset molecule atoms         :", subset.num_atoms)

## Selecting a subset

Pass a GENESIS selection string. The returned trajectory then contains only those
atoms, and `subset` is the matching reduced molecule.


In [ ]:
ca_trajs, ca_mol = genesis_exe.crd_convert(
    mol,
    trj_files=[str(BPTI_DCD)],
    trj_format="DCD",
    trj_type="COOR+BOX",
    selection="an:CA",
)
print(f"CA-only trajectory: {ca_trajs[0].coords.shape}, subset molecule: {ca_mol.num_atoms} atoms")

## Fitting at load time

Supplying `fitting_selection` + `fitting_method` superimposes every frame onto the
reference before returning. `fitting_method` is case-insensitive; common choices
are `"TR+ROT"` (translation + rotation), `"TR"`, and `"TR+ZROT"`.


In [ ]:
fit_trajs, _ = genesis_exe.crd_convert(
    mol,
    trj_files=[str(BPTI_DCD)],
    trj_format="DCD",
    trj_type="COOR+BOX",
    selection="an:CA",
    fitting_selection="an:CA",
    fitting_method="tr+rot",   # case-insensitive
)
print("fitted coords are finite:", np.all(np.isfinite(fit_trajs[0].coords)))

## Centering

`centering=True` moves a chosen group's center of mass to `center_coord`. This is
handy for membrane systems; here we center the C&alpha; atoms at the origin.


In [ ]:
cen_trajs, _ = genesis_exe.crd_convert(
    mol,
    trj_files=[str(BPTI_DCD)],
    trj_format="DCD",
    trj_type="COOR+BOX",
    selection="an:CA",
    centering=True,
    center_coord=(0.0, 0.0, 0.0),
)
com = cen_trajs[0].coords.mean(axis=1)   # per-frame center
print("per-frame center magnitude (should be near 0):", np.linalg.norm(com, axis=1).max())

## Plot the box size

`trj_type="COOR+BOX"` also returns the periodic box, which is useful for checking
NPT equilibration.


In [ ]:
import plotly.io as pio
import plotly.graph_objects as go
pio.renderers.default = "notebook"

# pbc_boxes is (nframe, 3, 3); the diagonal holds the box edge lengths.
box = np.diagonal(trajs[0].pbc_boxes, axis1=1, axis2=2)   # (nframe, 3)
colors = {"x": "#4C72B0", "y": "#DD8452", "z": "#55A868"}
fig = go.Figure()
for i, axis in enumerate("xyz"):
    fig.add_trace(go.Scatter(
        y=box[:, i], mode="lines+markers", name=f"box {axis}",
        line=dict(color=colors[axis], width=2.5),
        marker=dict(size=7, color=colors[axis], line=dict(width=1, color="white")),
    ))
fig.update_layout(
    title=dict(text="<b>Periodic box dimensions</b>", font=dict(size=18)),
    xaxis_title="Frame", yaxis_title="Length (&#8491;)",
    template="plotly_white",
    font=dict(family="Inter, Helvetica, Arial, sans-serif", size=13, color="#333"),
    hovermode="x unified", legend=dict(orientation="h", y=1.02, yanchor="bottom"),
    margin=dict(l=60, r=30, t=60, b=50), height=380,
)
fig

Next: for trajectories too large to hold in memory, see **[Lazy loading](02b_lazy_loading.ipynb)**.